In [35]:
# from ollama import chat

# # Initialize an empty message history
# messages = []
# while True:
#     user_input = input('Chat with history: ')
#     if user_input.lower() == 'exit':
#         break
#     # Get streaming response while maintaining conversation history
#     response_content = ""
#     for chunk in chat(
#         'deepseek-r1:1.5b',
#         messages=messages + [
#             {'role': 'system', 'content': 'You are a helpful assistant. You only give a short sentence by answer.'},
#             {'role': 'user', 'content': user_input},
#         ],
#         stream=True
#     ):
#         if chunk.message:
#             response_chunk = chunk.message.content
#             print(response_chunk, end='', flush=True)
#             response_content += response_chunk
#     # Add the exchange to the conversation history
#     messages += [
#         {'role': 'user', 'content': user_input},
#         {'role': 'assistant', 'content': response_content},
#     ]
#     print('\n')  # Add space after response

## Chargement des documents

In [36]:
# Configuration des chemins
DATA_FOLDER = "data"
files = [
    "Convention collective - services alimentaires.pdf",
    "Hébergement - Critères du service d'entretien ménager dans un hôtel 4 diamants.pdf",
    "Membres du comité SST.pdf",
    "politiques_hotel_de_la_promenade_document.pdf",
    "Principes de vie internes.pdf",
    "Processus de Gestion des reservations.pdf"
]

def get_document_category(filename):
    """Assigne une catégorie basée sur le nom du fichier pour le filtrage RAG."""
    if "Convention" in filename: return "RH - Légal"
    if "politiques" in filename: return "Opérations - Manuel"
    if "SST" in filename: return "Sécurité - Santé"
    if "Critères" in filename: return "Qualité - Service"
    return "Interne - Culture"


## Traitement des documents

In [37]:
import os
import re
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def clean_text(text):
    # 1. On protège les accents avant toute manipulation si nécessaire
    # Mais ici on va surtout nettoyer les espaces doubles et les sauts de ligne incohérents
    text = re.sub(r'[ \t]+', ' ', text)
    
    # 2. Supprimer les en-têtes répétitifs (Hôtel de la Promenade) 
    # seulement s'ils sont seuls sur une ligne
    lines = text.split('\n')
    cleaned_lines = [l for l in lines if "PROMENADE" not in l.upper() or len(l) > 30]
    text = "\n".join(cleaned_lines)
    
    # 3. Recoller les mots coupés par un saut de ligne (ex: "pré-\namble")
    text = re.sub(r'(\w)-\n(\w)', r'\1\2', text)
    
    return text.strip()

In [38]:
def get_section_title(text):
    """Détermine dynamiquement la section du chunk."""
    # Cherche "ARTICLE X", "CHAPITRE X" ou "Annexe X"
    match = re.search(r'(ARTICLE\s+\d+|CHAPITRE\s+\d+|ANNEXE\s+[A-Z])', text, re.IGNORECASE)
    if match:
        return match.group(0).upper()
    return None

## Chunking

In [39]:
from langchain_core.documents import Document

def process_hotel_docs(files):
    all_chunks = []
    
    # Splitter pour les gros documents
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
        separators=["\nARTICLE ", "\nCHAPITRE ", r"\n\d+\. ", "\n\n", "\n", ". "],
        keep_separator=True
    )

    for file_name in files:
        file_path = f"data/{file_name}"
        loader = PyPDFLoader(file_path)
        pages = loader.load()
        
        # 1. On regroupe tout le texte du fichier pour évaluer sa taille
        full_text = "\n".join([page.page_content for page in pages])
        cleaned_content = clean_text(full_text)
        
        category = get_document_category(file_name)
        
        # --- LOGIQUE OPTION B : TAILLE CONDITIONNELLE ---
        # Si le document est petit (ex: moins de 2000 caractères), on ne le coupe pas.
        # Cela garantit que l'adresse email et les membres du SST restent ensemble.
        if len(cleaned_content) < 2000:
            metadata = {
                "source": file_name,
                "categorie": category,
                "section": "Document Complet",
                "page": "1+",
                "type": "petit_doc_integral"
            }
            # On injecte le titre du document au début du texte pour le retriever
            final_content = f"DOCUMENT: {file_name}\n{cleaned_content}"
            all_chunks.append(Document(page_content=final_content, metadata=metadata))
            print(f"Traitement : {file_name} gardé en un seul bloc.")
            
        else:
            # Pour les gros documents (Convention, Politiques), on découpe normalement
            print(f"Traitement : {file_name} découpé en chunks.")
            chunks = text_splitter.split_text(cleaned_content)
            
            last_section = "Introduction"
            for i, chunk in enumerate(chunks):
                section = get_section_title(chunk)
                if section: last_section = section
                
                metadata = {
                    "source": file_name,
                    "categorie": category,
                    "section": last_section,
                    "page": "Calculée", # On peut affiner ici
                    "type": "chunk_sequence"
                }
                
                # Injection de contexte : on force le nom du doc et la section dans le texte
                contextualized_content = f"SOURCE: {file_name} | SECTION: {last_section}\n{chunk}"
                
                all_chunks.append(Document(page_content=contextualized_content, metadata=metadata))
                
    return all_chunks

In [40]:
rag_ready_data = process_hotel_docs(files)
print(f"Nombre total de chunks : {len(rag_ready_data)}")
print(f"Exemple : {rag_ready_data[2].page_content[:200]}")
print(f"Métadonnées : {rag_ready_data[2].metadata}")

Traitement : Convention collective - services alimentaires.pdf découpé en chunks.
Traitement : Hébergement - Critères du service d'entretien ménager dans un hôtel 4 diamants.pdf gardé en un seul bloc.
Traitement : Membres du comité SST.pdf gardé en un seul bloc.
Traitement : politiques_hotel_de_la_promenade_document.pdf découpé en chunks.
Traitement : Principes de vie internes.pdf découpé en chunks.
Traitement : Processus de Gestion des reservations.pdf gardé en un seul bloc.
Nombre total de chunks : 89
Exemple : SOURCE: Convention collective - services alimentaires.pdf | SECTION: ARTICLE 1
ARTICLE 1 - PRÉAMBULE 
1.1 Objectifs de la présente entente 
La présente convention collective a pour but : 
a) D’établir
Métadonnées : {'source': 'Convention collective - services alimentaires.pdf', 'categorie': 'RH - Légal', 'section': 'ARTICLE 1', 'page': 'Calculée', 'type': 'chunk_sequence'}


## Embeddings et Vectorisation

In [41]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

# 1. Plus besoin de boucle de conversion ! 
# rag_ready_data contient déjà des objets Document avec page_content et metadata.
documents_for_chroma = rag_ready_data 

# 2. Configuration des embeddings
embeddings_model = OllamaEmbeddings(model="nomic-embed-text")

# 3. Création du vectorstore avec persistance
# J'ajoute le dossier de persistance pour éviter de recalculer à chaque lancement
vectorstore = Chroma.from_documents(
    documents=documents_for_chroma, 
    embedding=embeddings_model
)

## Prompt systeme et chaine de generation

In [42]:
# On affine le prompt pour le personnel de l'hôtel
SYSTEM_PROMPT = """Tu es l'Assistant RH et Opérations de l'Hôtel de la Promenade.
Ton rôle est d'aider les employés à trouver des informations fiables dans les documents internes.

RÈGLES DE RÉPONSE :
1. Utilise UNIQUEMENT le CONTEXTE fourni.
2. Cite TOUJOURS la source (Nom du fichier et Page) à la fin de ta réponse.
3. Si l'information concerne une procédure, donne les étapes clairement (liste à puces).
4. Si l'information n'est pas dans le contexte, réponds : "Je suis désolé, je ne trouve pas cette information dans les manuels officiels. Veuillez contacter votre superviseur ou les RH."
5. Reste professionnel, bienveillant et factuel.
"""

In [43]:
from ollama import chat

def format_docs_with_metadata(docs):
    """
    Formate les chunks de manière structurée pour optimiser la réponse du LLM
    en incluant la hiérarchie des sections.
    """
    if not docs:
        return "AUCUN CONTEXTE DISPONIBLE."

    context_blocks = []
    
    for i, doc in enumerate(docs, 1):
        # Extraction sécurisée des métadonnées
        source = doc.metadata.get('source', 'Document inconnu')
        page = doc.metadata.get('page', 'N/A')
        cat = doc.metadata.get('categorie', 'Information générale')
        section = doc.metadata.get('section', 'Section non spécifiée')
        
        # Formatage du bloc avec une structure claire
        block = (
            f"--- EXTRAIT {i} ---\n"
            f"PROVENANCE : {source}\n"
            f"CONTEXTE : {cat} | {section}\n"
            f"EMPLACEMENT : Page {page}\n"
            f"CONTENU : {doc.page_content.strip()}\n"
            f"----------------"
        )
        context_blocks.append(block)
    
    return "\n\n".join(context_blocks)

In [44]:
def smart_retrieve(question, vectorstore):
    q = question.lower()

    # ── Sécurité - Santé ──────────────────────────────────────────
    # Source : Membres_du_comité_SST.pdf (bloc entier)
    if any(w in q for w in [
        "sst", "ssbe", "santé", "sécurité", "securite",
        "bien-être", "bien-etre", "prévention", "prevention",
        "accident", "risque", "danger",
        "carole ritz", "chris evans", "clara dupéré", "clara dupere",
        "david martin", "hi-selon kang", "justin lemieux",
        "comité de la santé", "comite de la sante",
        "ssbe@delapromenade"
    ]):
        filtre = {"categorie": "Sécurité - Santé"}

    # ── Qualité - Service ─────────────────────────────────────────
    # Source : Critères 4 diamants (bloc entier)
    elif any(w in q for w in [
        "diamant", "entretien ménager", "entretien menager",
        "couverture", "couvre-lit", "oreiller", "serviette",
        "cendrier", "corbeille", "rideaux", "éclairage", "eclairage",
        "radio", "préposé", "prepose", "ménage chambre", "menage chambre",
        "trois coups", "service de couverture"
    ]):
        filtre = {"categorie": "Qualité - Service"}

    # ── Interne - Culture ─────────────────────────────────────────
    # Source : Principes de vie internes (chunks normaux)
    # 10 principes : Esprit d'équipe, Professionnalisme,
    # Engagement envers l'excellence, Préservation de l'image immaculée,
    # Adaptabilité et flexibilité, Communication efficace,
    # Apprentissage continu, Respect et diversité,
    # Responsabilité environnementale, Équilibre vie pro/perso
    # Signé : Saïd Bensouri, Directeur général
    elif any(w in q for w in [
        # Titre et intro du document
        "principes de vie", "principes internes", "vie interne",
        "said bensouri", "saïd bensouri",
        # Principe 1 - Esprit d'équipe
        "esprit d'équipe", "esprit d equipe", "collaboration",
        "harmonie", "contribution individuelle", "succès collectif",
        "communication ouverte", "respect mutuel",
        # Principe 2 - Professionnalisme
        "professionnalisme", "attitude professionnelle",
        "éthique", "ethique", "discrétion", "discretion",
        "confidentialité", "confidentialite",
        # Principe 3 - Engagement envers l'excellence
        "excellence", "haute qualité", "haute qualite",
        "service exceptionnel", "expérience mémorable", "experience memorable",
        "empathie", "anticiper",
        # Principe 4 - Préservation de l'image immaculée
        "image immaculée", "image immaculee", "propreté", "proprete",
        "apparence impeccable", "espaces", "installations", "équipements",
        "coulisses", "présentable",
        # Principe 5 - Adaptabilité et flexibilité
        "adaptabilité", "adaptabilite", "flexibilité", "flexibilite",
        "changement", "nouvelles situations", "flexible",
        "soutien dans différents", "soutien dans differents",
        # Principe 6 - Communication efficace
        "communication efficace", "communication claire",
        "écoute active", "ecoute active", "suggestions",
        "préoccupations", "preoccupations", "informations pertinentes",
        # Principe 7 - Apprentissage continu
        "apprentissage continu", "développement personnel",
        "developpement personnel", "formation", "compétences",
        "competences", "plein potentiel",
        # Principe 8 - Respect et diversité
        "diversité", "diversite", "inclusif", "inclusion",
        "équité", "equite", "origine ethnique", "orientation sexuelle",
        # Principe 9 - Responsabilité environnementale
        "responsabilité environnementale", "responsabilite environnementale",
        "environnement", "conservation des ressources",
        "réduction des déchets", "reduction des dechets",
        "initiatives durables",
        # Principe 10 - Équilibre vie pro/perso
        "équilibre", "equilibre", "vie professionnelle", "vie personnelle",
        "santé mentale", "sante mentale", "bien-être équipe",
        "limites individuelles"
    ]):
        filtre = {"categorie": "Interne - Culture"}

    # ── Interne - Culture (Processus réservations) ────────────────
    # Source : Processus_de_Gestion_des_reservations.pdf (bloc entier)
    # Contenu : processus défaillant, acteurs Client + Réceptionniste,
    # 6 étapes, planning manuel, promesse verbale, pas de confirmation écrite
    elif any(w in q for w in [
        "réservation", "reservation", "réserver", "reserver",
        "processus", "étapes réservation", "etapes reservation",
        "réceptionniste", "receptionniste",
        "planning", "disponibilité", "disponibilite",
        "confirmation écrite", "confirmation ecrite",
        "promesse verbale", "note temporaire",
        "formulaire", "saisie", "demande de réservation",
        "demande de reservation", "défaillant", "defaillant"
    ]):
        filtre = {"categorie": "Interne - Culture"}

    # ── RH - Légal ────────────────────────────────────────────────
    # Source : Convention collective (chunks par ARTICLE)
    elif any(w in q for w in [
        # Article 1
        "convention", "préambule", "preambule", "entente",
        "masculin", "genre neutre", "définition", "definition",
        # Article 2
        "discrimination", "charte", "coercition", "contrainte",
        # Article 3
        "grève", "greve", "lock-out", "piquetage",
        # Article 4
        "syndicat", "syndical", "négociation", "negociation",
        "sous-traitance", "représentant syndical",
        # Article 5
        "cotisation", "adhésion", "adhesion",
        # Article 6
        "gérance", "gerance", "droit de gérance",
        # Article 7
        "disciplinaire", "probation", "licenciement",
        "congédiement", "suspension",
        # Article 8
        "grief", "griefs", "divergence", "violation",
        # Article 9
        "arbitrage", "arbitre",
        # Article 11
        "uniforme", "blanchissage", "chaussures de sécurité",
        # Article 12
        "ancienneté", "anciennete", "liste d ancienneté",
        # Article 13
        "mutation", "promotion", "affichage de poste",
        # Article 14-15
        "heures de travail", "heures supplémentaires",
        "horaire", "pause", "repos", "quart de travail",
        # Article 16-17
        "salaire", "taux horaire", "pourboire", "paie",
        # Article 18-21
        "congé férié", "conge ferie", "vacances annuelles",
        "congé de maladie", "congé de deuil",
        # Article 22-23
        "avantages sociaux", "assurance collective",
        # Annexe A
        "aide-cuisinier", "chef cuisinier", "pâtissier",
        "patissier", "plongeur", "serveur", "caissier"
    ]):
        filtre = {"categorie": "RH - Légal"}

    # ── Opérations - Manuel ───────────────────────────────────────
    # Source : politiques_hotel (chunks thématiques)
    elif any(w in q for w in [
        # Section 1
        "slogan", "présentation", "presentation",
        "ottawa", "promenade de l aviation", "k1k 4r3",
        # Section 2
        "mission", "vision", "valeurs",
        "respect", "excellence", "intégrité", "integrite",
        "durabilité", "durabilite",
        # Section 3
        "contact", "téléphone", "telephone", "courriel",
        "conciergerie", "direction générale",
        # Section 4
        "check-in", "check-out", "annulation", "no-show",
        "early check", "late check",
        # Section 5
        "chambre classique", "chambre deluxe", "suite junior",
        "suite senior", "suite exécutive", "nespresso", "minibar",
        # Section 6
        "bistro", "restaurant", "menu", "petit-déjeuner",
        "brunch", "room service", "service aux chambres",
        "allergie", "alcool", "débouchonnage",
        # Section 7
        "salle bourgeois", "salle racine", "salle poirier",
        "réunion", "congrès", "congres",
        # Section 8
        "spa", "oasis urbaine", "massage", "soin", "bain thermal",
        # Section 9
        "stationnement", "valet", "vélo", "navette", "aéroport",
        "borne de recharge",
        # Section 10
        "animaux de compagnie", "chien d assistance",
        # Section 11
        "code vestimentaire", "veston", "tablier", "tunique",
        "tatouage", "pointage", "retard",
        # Section 12
        "urgence", "évacuation", "dea", "défibrillateur",
        "point de rassemblement", "parc de la confédération",
        # Section 13
        "forfait couette", "forfait famille", "forfait romantique",
        "eco-vert", "promenade or",
        # Section 14-15
        "haccp", "fournisseur", "compostage", "événement", "banquet",
        # Section 16
        "lost and found", "objet trouvé", "formulaire incident"
    ]):
        filtre = {"categorie": "Opérations - Manuel"}

    else:
        filtre = None  # fallback MMR global k=8

    retriever = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={
            "k": 6 if filtre else 8,
            "fetch_k": 15 if filtre else 20,
            "lambda_mult": 0.6,
            **({"filter": filtre} if filtre else {})
        }
    )
    return retriever.invoke(question)

In [45]:
# Initialisation
messages = []

while True:
    user_input = input("\nQuestion Employé > ").strip()
    if not user_input: continue
    if user_input.lower() in ["exit", "quitter"]: break

    # 1. Récupération des chunks (On s'assure que le retriever répond)
    retrieved_docs = smart_retrieve(user_input, vectorstore)
    
    if not retrieved_docs:
        print("Assistant: Je n'ai trouvé aucune information officielle à ce sujet.")
        continue

    # 2. Formatage du contexte
    context_text = format_docs_with_metadata(retrieved_docs)

    # 3. Prompt système mis à jour pour CHAQUE question
    # On place le contexte dans le 'system' pour que le LLM le traite en priorité
    current_system_prompt = f"{SYSTEM_PROMPT}\n\nCONTEXTE :\n{context_text}"

    # Construction du message (On garde un historique court pour la fluidité)
    api_messages = [{'role': 'system', 'content': current_system_prompt}]
    for msg in messages[-4:]: # Ajout de l'historique conversationnel
        api_messages.append(msg)
    api_messages.append({'role': 'user', 'content': user_input})

    print(f"Utilisateur: {user_input}")
    print(f"Assistant: ", end="", flush=True)
    
    full_response = ""
    
    # 4. Boucle de streaming corrigée
    try:
        response_generator = chat(model="llama3.1:8b", messages=api_messages, stream=True)
        
        for chunk in response_generator:
            # Correction : Accès direct au contenu du message
            content = chunk['message']['content']
            
            if content:
                # Filtrage optionnel des balises de pensée
                clean_content = re.sub(r'<think>.*?</think>', '', content, flags=re.DOTALL)
                if "<think>" not in content: # Évite d'afficher le début d'une balise
                    print(clean_content, end="", flush=True)
                full_response += content

    except Exception as e:
        print(f"\nErreur de connexion avec Ollama : {e}")

    print("\n" + "-"*50)

    # 5. Archivage de la réponse (sans les balises think)
    final_answer = re.sub(r'<think>.*?</think>', '', full_response, flags=re.DOTALL).strip()
    
    if final_answer:
        messages.append({"role": "user", "content": user_input})
        messages.append({"role": "assistant", "content": final_answer})

Utilisateur: quel jour sommes-nous ?
Assistant: D'après les jours fériés listés dans l'extrait 1 (Convention collective - services alimentaires.pdf | PAGE CALCULÉE), on voit que le Jour de la famille est le 2e lundi du mois de février.

Je ne peux pas déterminer avec certitude la date exacte sans plus d'informations. Si vous voulez savoir si aujourd'hui, c'est un jour férié, alors je dois consulter d'autres documents internes pour avoir les informations les plus à jour. 

Si tu veux connaître la date ou si aujourd'hui est un jour férié, je peux te suggérer de contacter le chef d'unité responsable des RH ou le HR directement. 
(SOURCE: Convention collective - services alimentaires.pdf | SECTION: ARTICLE 18)
--------------------------------------------------
Utilisateur: Parle moi un peu de l'hotel La Promenade
Assistant: L'Hôtel de la Promenade ! C'est un établissement prestigieux situé dans un endroit splendide. D'après ce que j'ai pu voir dans les documents internes, voici quelques in